# 01. Two pointers

**Tier:** Deep. **Why you'd use this:** On a sorted list, one pointer starts at the left and one at the right. If the sum is too small, move the left pointer.

**After this lesson you can:** move the left or right pointer on a sorted list; find a target pair without a dict.
About 20 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. Why does moving the larger pointer down make sense on a sorted list?
2. When would you still want a hash map instead of two pointers?

<details><summary>Answers</summary>

For 01 sorted pair lesson: skim [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb) if needed.
</details>


## Recognize it

The list is sorted. You need a pair whose sum hits a target. The prompt mentions two ends, moving inward, or finding a pair without a hash map.

Signals: sorted order, a numeric target, and a pair (or a yes/no for a pair). A look-alike that is not this pattern: an unsorted list where you store complements in a dict — that is a hash map, and sorting first would change the indexes.

Also reject a contiguous-subarray reading. "Two numbers that add to target" on a sorted array is opposite-end pointers. "A slice whose sum hits target" is a window or a prefix map, not this inward walk.


## Why it works this way

On a sorted list the smallest unused value sits at the left and the largest at the right. Their sum is the extreme pair. If that sum is below the target, every pair that still uses the current left value with something smaller than the current right is even smaller, so those right values cannot help. Moving the left pointer up is safe. If the sum is above the target, every pair that still uses the current right value with something larger than the current left is even larger, so moving the right pointer down is safe. Each step throws away one value forever. The pointers meet after at most `n` moves, so time stays proportional to `n`. Extra memory is a handful of integers.

A nested double loop still works and needs no sorted order, but it compares about `n² / 2` pairs. Sorting first costs roughly `n log n`, then the two-pointer finish is linear. When the list arrives already sorted, you skip the sort cost. A hash map also finds a pair in expected linear time, but it stores up to `n` keys and does not need sorted order. Two pointers win when you must keep extra memory constant and the input is sorted (or sorting is allowed and indexes in the sorted copy are enough).

The invariant at every step: if a pair exists in the remaining slice, it still lies between `left` and `right`. That is why crossing the pointers means no pair exists. Empty and one-element lists fail the `left < right` guard immediately and return false.

The move is irreversible on purpose. You never walk a pointer backward because the sorted order already proved that the discarded side cannot form a hit with anything still in range. That proof is the whole reason the pattern exists: sorted order turns a two-dimensional search into a one-dimensional shrink.


## When to use it

Use opposite-end pointers when the list is sorted and the question is a pair sum (or a close cousin like closest pair). Use a hash map when the list is unsorted and you need original indexes. Use a nested loop only for tiny `n`. Do not force this pattern on a contiguous subarray problem — that is usually a sliding window or a prefix sum.

If the prompt forbids sorting and forbids extra memory, you are stuck with the nested scan. If they allow sorting but need original indexes afterward, sort indexes by value instead of sorting the values alone, or fall back to the hash map.


## Brute force, and why it's slow

Try every pair. Count how many sums you compute.


In [1]:
def has_pair_slow(nums, target):
    checks = 0
    for i in range(len(nums)):
        for j in range(i + 1, len(nums)):
            checks += 1
            if nums[i] + nums[j] == target:
                return True, checks
    return False, checks


for size in (40, 80, 160):
    data = list(range(size))
    print(size, has_pair_slow(data, data[0] + data[-1]))


40 (True, 39)
80 (True, 79)
160 (True, 159)


## Key insight

On a sorted list, a sum that is too small can only improve by moving the left pointer up; a sum that is too big can only improve by moving the right pointer down. One move discards one value forever.


## Worked example

Return whether any two values in a sorted list sum to the target. This returns a bool, not indexes.


In [2]:
def has_pair(nums: list[int], target: int) -> bool:
    # 1. Start at the two ends of a sorted list.
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        # 2. Move the end that can still hit the target.
        if total == target:
            return True
        if total < target:
            left += 1
        else:
            right -= 1
    return False


print(has_pair([1, 2, 4, 7], 6))
print(has_pair([1, 2, 4, 7], 20))
print(has_pair([5], 10))


True
False
False


## Trace

`[1, 2, 4, 7]`, target `6`:

| left | right | sum | move |
|---|---|---|---|
| 0 (1) | 3 (7) | 8 | too big → right− |
| 0 (1) | 2 (4) | 5 | too small → left+ |
| 1 (2) | 2 (4) | 6 | **hit** |


## Cost

Each step moves one pointer. They start `n - 1` apart and never move outward, so the loop runs at most `n - 1` times. Extra memory is constant. The nested scan pays quadratic pair checks on the same input. Sorting first, when needed, adds roughly `n log n` before the linear finish.


## Variants

Not solved here: [3Sum](../../challenges/problems/04-3sum.ipynb), [Container With Most Water](../../challenges/problems/05-container-with-most-water.ipynb), [Two Sum](../../challenges/problems/01-two-sum.ipynb).


## Common mistake

This bug is silent. The loop uses `left <= right`, so a single value can pair with itself when `2 * nums[left] == target`. There is no traceback. For `[3]` and target `6` the print is `True`, and the answer is `False`.


In [3]:
def has_pair_wrong(nums, target):
    left, right = 0, len(nums) - 1
    while left <= right:  # allows the same index twice
        total = nums[left] + nums[right]
        if total == target:
            return True
        if total < target:
            left += 1
        else:
            right -= 1
    return False


print(has_pair_wrong([3], 6))


True


The fix keeps `left < right` so the two indexes stay distinct. The worked example does that, so `[3]` with target `6` prints `False`.


In [4]:
print(has_pair([3], 6))


False


## Python idioms

Tuple unpacking sets both ends in one line. When you need the values, not the indexes, read `nums[left]` and `nums[right]` inside the loop instead of building a second structure.


In [5]:
def pair_values(nums: list[int], target: int) -> tuple[int, int] | None:
    left, right = 0, len(nums) - 1
    while left < right:
        total = nums[left] + nums[right]
        if total == target:
            return nums[left], nums[right]
        if total < target:
            left += 1
        else:
            right -= 1
    return None


print(pair_values([1, 2, 4, 7], 6))


(2, 4)


## Recap

- Sorted order lets the two ends close in.
- Too small moves left. Too big moves right.
- Stop when the pointers meet; they must stay on different indexes.

Next: [practice](01-sorted-pair-practice.ipynb).
